<a href="https://colab.research.google.com/github/milka611/what-to-cook/blob/main/resume_yolo26n_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

folder = Path("/content/drive/MyDrive/what_to_cook_ai")

if not folder.is_dir():
    raise FileNotFoundError(
        f"Папка не найдена: {folder}\n"
        "Проверьте её название и расположение на Google Диске."
    )

extensions = {
    ".pt", ".zip", ".rar", ".7z", ".tar", ".gz",
    ".yaml", ".yml", ".json", ".csv", ".ipynb", ".py", ".txt"
}

found = []
for path in sorted(folder.rglob("*")):
    if path.is_file() and path.suffix.lower() in extensions:
        size_mb = path.stat().st_size / (1024 ** 2)
        found.append(
            f"{size_mb:10.2f} МБ | {path.relative_to(folder)}"
        )

report = "\n".join(found) or "Подходящие файлы не найдены."
print(report)
print("\nПроверка завершена. Обучение НЕ запускалось.")

Mounted at /content/drive
    711.64 МБ | Food Ingredients Detection.v3i.yolov8.zip
    207.08 МБ | Fruits-And-Vegetables-Detection-Dataset-main.zip
   1369.46 МБ | archive(1).zip
     20.36 МБ | best.pt
      5.21 МБ | best_test.pt
    358.11 МБ | groceries-object-detection-dataset-master.zip
     20.36 МБ | last.pt

Проверка завершена. Обучение НЕ запускалось.


In [ ]:
from pathlib import Path, PurePosixPath
from collections import Counter
import zipfile
import pickletools

folder = Path("/content/drive/MyDrive/what_to_cook_ai")
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for archive in sorted(folder.glob("*.zip")):
    print(f"\n{'=' * 65}\nАРХИВ: {archive.name}")

    try:
        with zipfile.ZipFile(archive) as z:
            members = [i for i in z.infolist() if not i.is_dir()]
            images = [
                i for i in members
                if PurePosixPath(i.filename).suffix.lower()
                in image_extensions
            ]
            labels = [
                i for i in members
                if PurePosixPath(i.filename).suffix.lower() == ".txt"
            ]

            print("Изображений:", len(images))
            print("TXT-файлов (не обязательно меток):", len(labels))

            # Показываем расположение изображений, чтобы увидеть splits.
            directories = Counter(
                str(PurePosixPath(i.filename).parent)
                for i in images
            )
            print("Каталоги изображений, до 25:")
            for directory, count in directories.most_common(25):
                print(f"  {count:6d} | {directory}")
            if len(directories) > 25:
                print("  ... остальные каталоги не показаны")

            descriptions = [
                i for i in members
                if (
                    PurePosixPath(i.filename).suffix.lower()
                    in {".yaml", ".yml", ".names"}
                    or PurePosixPath(i.filename).name.lower()
                    in {"classes.txt", "obj.names"}
                )
            ]

            print("Файлы описания классов:", len(descriptions))
            for info in descriptions[:15]:
                print(f"\n--- {info.filename} ---")
                if info.file_size > 64_000:
                    print("Пропущен: размер больше 64 КБ")
                    continue
                text = z.read(info).decode("utf-8-sig", errors="replace")
                print(text[:20_000])
                if len(text) > 20_000:
                    print("[вывод сокращён]")

            scripts = [
                i.filename for i in members
                if PurePosixPath(i.filename).suffix.lower()
                in {".py", ".ipynb"}
            ]
            print("\nСкрипты и блокноты внутри архива, до 20:")
            print("\n".join(scripts[:20]) or "Не найдены")

    except zipfile.BadZipFile:
        print("ОШИБКА: архив не распознаётся как ZIP")


print(f"\n{'=' * 65}\nМЕТАДАННЫЕ last.pt")
checkpoint = folder / "last.pt"

# Читаем pickle-инструкции как данные; pickle.load / torch.load не вызываются.
if zipfile.is_zipfile(checkpoint):
    with zipfile.ZipFile(checkpoint) as z:
        candidates = [
            i for i in z.infolist()
            if i.filename.endswith("/data.pkl")
            or i.filename == "data.pkl"
        ]

        if len(candidates) != 1:
            print("Не найден однозначный data.pkl:", len(candidates))
        elif candidates[0].file_size > 16 * 1024 * 1024:
            print("data.pkl слишком велик для этой проверки")
        else:
            strings = []
            for op, value, position in pickletools.genops(
                z.read(candidates[0])
            ):
                if isinstance(value, str) and op.name in {
                    "UNICODE", "BINUNICODE",
                    "SHORT_BINUNICODE", "BINUNICODE8"
                }:
                    strings.append(value)

            for key in (
                "version", "date", "model", "data",
                "epochs", "optimizer", "train_args"
            ):
                indexes = [
                    i for i, value in enumerate(strings) if value == key
                ]
                for i in indexes[:2]:
                    print(
                        f"{key}: соседние текстовые значения =",
                        strings[i + 1:i + 5]
                    )

            print("\nУпоминания архитектуры и библиотек, до 30:")
            mentions = list(dict.fromkeys(
                s for s in strings
                if any(word in s.lower() for word in (
                    "yolo26", "ultralytics", ".yaml"
                ))
            ))
            for value in mentions[:30]:
                print(value[:400])
else:
    print("Старый формат checkpoint: нужна отдельная проверка.")

print(
    "\nЭто предварительная инспекция. Соседние строки не доказывают "
    "значения параметров checkpoint.\n"
    "Обучение НЕ запускалось."
)


АРХИВ: Food Ingredients Detection.v3i.yolov8.zip
Изображений: 24001
TXT-файлов (не обязательно меток): 24003
Каталоги изображений, до 25:
   21544 | train/images
    1746 | valid/images
     711 | test/images
Файлы описания классов: 1

--- data.yaml ---
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 13
names: ['Beef', 'Cabbage', 'Carrot', 'Chicken', 'Cucumber', 'Egg', 'Eggplant', 'Leek', 'Onion', 'Pork', 'Potato', 'Radish', 'Tomato']

roboflow:
  workspace: nutrilens-qvsz6
  project: food-ingredients-detection-nxe34
  version: 3
  license: CC BY 4.0
  url: https://universe.roboflow.com/nutrilens-qvsz6/food-ingredients-detection-nxe34/dataset/3

Скрипты и блокноты внутри архива, до 20:
Не найдены

АРХИВ: Fruits-And-Vegetables-Detection-Dataset-main.zip
Изображений: 214
TXT-файлов (не обязательно меток): 180
Каталоги изображений, до 25:
     180 | Fruits-And-Vegetables-Detection-Dataset-main/LVIS_Fruits_And_Vegetables/images/test
      20 | Fruits-And-Vegetables-D

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "ultralytics==8.4.175"
    ],
    check=True
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'ultralytics==8.4.175'], returncode=0)

In [ ]:
from pathlib import Path, PurePosixPath
from collections import Counter
import json
import zipfile
import torch
import ultralytics

folder = Path("/content/drive/MyDrive/what_to_cook_ai")
assert ultralytics.__version__ == "8.4.175"

# Только чтение: исходные веса не перезаписываются.
ckpt = torch.load(
    folder / "last.pt",
    map_location="cpu",
    weights_only=False
)

print("=== CHECKPOINT ===")
print("Версия:", ckpt.get("version"))
print("Дата:", ckpt.get("date"))
print("epoch, с нуля:", ckpt.get("epoch"))

epoch = ckpt.get("epoch")
if isinstance(epoch, int):
    print("Завершено эпох:", epoch + 1)
    print("Следующая эпоха:", epoch + 2)

optimizer = ckpt.get("optimizer")
print("Optimizer присутствует:", optimizer is not None)
if isinstance(optimizer, dict):
    print("Состояний параметров:", len(optimizer.get("state", {})))
    print("Групп параметров:", len(optimizer.get("param_groups", [])))

args = ckpt.get("train_args", {})
print("\n=== ПАРАМЕТРЫ ОБУЧЕНИЯ ===")
keys = (
    "model", "task", "data", "epochs", "batch", "imgsz",
    "optimizer", "lr0", "lrf", "seed", "deterministic",
    "fraction", "single_cls", "classes", "freeze",
    "rect", "close_mosaic", "save", "save_period",
    "project", "name"
)
for key in keys:
    print(f"{key}: {args.get(key)!r}")

net = ckpt.get("ema")
if net is None:
    net = ckpt.get("model")
assert net is not None, "В checkpoint не найдена модель"

raw_names = net.names
if isinstance(raw_names, dict):
    names = [raw_names[i] for i in range(len(raw_names))]
else:
    names = list(raw_names)

print("\n=== КЛАССЫ CHECKPOINT В ИСХОДНОМ ПОРЯДКЕ ===")
print(json.dumps(
    dict(enumerate(names)),
    ensure_ascii=False,
    indent=2
))
print("Количество классов:", len(names))
print("Уникальных названий:", len(set(names)))

head = net.model[-1]
print("\n=== АРХИТЕКТУРА ===")
print("Detection head:", type(head).__name__)
print("Head nc:", getattr(head, "nc", None))
architecture = getattr(net, "yaml", {})
for key in ("yaml_file", "scale", "nc", "scales"):
    print(f"{key}: {architecture.get(key)!r}")

checks = {
    "Версия 8.4.175": ckpt.get("version") == "8.4.175",
    "Завершена третья эпоха": epoch == 2,
    "Исходный бюджет — 20 эпох": args.get("epochs") == 20,
    "Есть состояние optimizer": (
        isinstance(optimizer, dict)
        and bool(optimizer.get("state"))
    ),
    "95 классов": len(names) == 95,
    "Head nc = 95": getattr(head, "nc", None) == 95,
}
print("\n=== ПРЕДВАРИТЕЛЬНЫЕ ПРОВЕРКИ RESUME ===")
for title, passed in checks.items():
    print(("OK" if passed else "НЕ ПОДТВЕРЖДЕНО"), "|", title)

print("\n=== GROCERIES: ФОРМАТ АННОТАЦИЙ ===")
archive = folder / "groceries-object-detection-dataset-master.zip"

with zipfile.ZipFile(archive) as z:
    members = [i for i in z.infolist() if not i.is_dir()]
    extensions = Counter(
        PurePosixPath(i.filename).suffix.lower() or "(без расширения)"
        for i in members
    )
    print("Расширения:", dict(extensions))

    image_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    candidates = [
        i for i in members
        if PurePosixPath(i.filename).suffix.lower() not in image_ext
    ]

    print("\nНеизобразительные файлы, первые 40:")
    for info in candidates[:40]:
        print(info.filename, "| байт:", info.file_size)

    # Читаем несколько небольших примеров, ничего не исполняем.
    examples = [
        i for i in candidates
        if PurePosixPath(i.filename).suffix.lower()
        in {".xml", ".json", ".csv", ".txt", ".py", ".md"}
        and i.file_size <= 64_000
    ]

    print("\nПримеры содержимого, до 5 файлов:")
    for info in examples[:5]:
        print("\n---", info.filename, "---")
        text = z.read(info).decode("utf-8-sig", errors="replace")
        print(text[:5_000])

print("\nОбучение НЕ запускалось. Файлы не изменены.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
=== CHECKPOINT ===
Версия: 8.4.175
Дата: 2026-10-10T13:36:29.591905+00:00
epoch, с нуля: 2
Завершено эпох: 3
Следующая эпоха: 4
Optimizer присутствует: True
Состояний параметров: 366
Групп параметров: 3

=== ПАРАМЕТРЫ ОБУЧЕНИЯ ===
model: 'yolo26n.pt'
task: 'detect'
data: '/content/food_three_full/data.yaml'
epochs: 20
batch: 8
imgsz: 640
optimizer: 'AdamW'
lr0: 0.000101
lrf: 0.01
seed: 0
deterministic: True
fraction: 1.0
single_cls: False
classes: None
freeze: None
rect: False
close_mosaic: 10
save: True
save_period: -1
project: '/content/food_training'
name: '95_products'

=== КЛАССЫ CHECKPOINT В ИСХОДНОМ ПОРЯДКЕ ===
{
  "0": "Beef",
  "1": "Cabbage",
  "2": "Carrot",
  "3": "Chicken",
  "4": "Cuc

In [ ]:
from pathlib import Path
import json

drive_root = Path("/content/drive/MyDrive")

# Папка проекта и стандартные/локализованные папки Colab.
search_dirs = [drive_root / "what_to_cook_ai"]
for p in drive_root.iterdir():
    if p.is_dir() and (
        "colab" in p.name.lower()
        or "блокнот" in p.name.lower()
    ):
        search_dirs.append(p)

notebooks = set(drive_root.glob("*.ipynb"))
for directory in search_dirs:
    if directory.is_dir():
        notebooks.update(directory.rglob("*.ipynb"))

keywords = (
    "food_three_full",
    "95_products",
    "groceries-object-detection",
    "Food Ingredients Detection",
    "archive(1).zip",
)

print("Проверяемые папки:")
for directory in search_dirs:
    print(" •", directory)
print("Также проверяются блокноты непосредственно в MyDrive.")
print("Найдено блокнотов:", len(notebooks))

budget = 18_000
matched = 0

for path in sorted(notebooks):
    if path.stat().st_size > 20 * 1024 * 1024:
        print("Пропущен большой блокнот:", path.relative_to(drive_root))
        continue

    try:
        notebook = json.loads(path.read_text(encoding="utf-8-sig"))
    except Exception as exc:
        print("Не прочитан:", path.name, type(exc).__name__)
        continue

    hits = []
    for number, cell in enumerate(notebook.get("cells", []), start=1):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)

        if any(word.lower() in source.lower() for word in keywords):
            hits.append((number, source))

    if not hits:
        continue

    matched += 1
    print("\nБЛОКНОТ:", path.relative_to(drive_root))
    print("Подходящие ячейки:", [number for number, _ in hits])

    for number, source in hits:
        if budget <= 0:
            break
        fragment = source[:min(2500, budget)]
        print(f"\n--- Ячейка {number} ---\n{fragment}")
        if len(fragment) < len(source):
            print("[Фрагмент сокращён]")
        budget -= len(fragment)

print("\nБлокнотов с совпадениями:", matched)
print("Обучение НЕ запускалось.")

Проверяемые папки:
 • /content/drive/MyDrive/what_to_cook_ai
 • /content/drive/MyDrive/Colab Notebooks
Также проверяются блокноты непосредственно в MyDrive.
Найдено блокнотов: 6

БЛОКНОТ: Colab Notebooks/resume_yolo26n_colab.ipynb
Подходящие ячейки: [4]

--- Ячейка 4 ---
from pathlib import Path, PurePosixPath
from collections import Counter
import json
import zipfile
import torch
import ultralytics

folder = Path("/content/drive/MyDrive/what_to_cook_ai")
assert ultralytics.__version__ == "8.4.175"

# Только чтение: исходные веса не перезаписываются.
ckpt = torch.load(
    folder / "last.pt",
    map_location="cpu",
    weights_only=False
)

print("=== CHECKPOINT ===")
print("Версия:", ckpt.get("version"))
print("Дата:", ckpt.get("date"))
print("epoch, с нуля:", ckpt.get("epoch"))

epoch = ckpt.get("epoch")
if isinstance(epoch, int):
    print("Завершено эпох:", epoch + 1)
    print("Следующая эпоха:", epoch + 2)

optimizer = ckpt.get("optimizer")
print("Optimizer присутствует:", optim

In [ ]:
from pathlib import Path
import json
import hashlib

path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "what_to_cook_training.ipynb"
)
assert path.is_file(), f"Не найден: {path}"

raw = path.read_bytes()
notebook = json.loads(raw.decode("utf-8-sig"))

print("ИСХОДНЫЙ БЛОКНОТ:", path.name)
print("SHA-256:", hashlib.sha256(raw).hexdigest())

# Нумерация соответствует предыдущему выводу.
# Ячейка 27 с запуском обучения сюда не входит.
for number, cell in enumerate(notebook.get("cells", []), start=1):
    if number >= 27:
        break
    if cell.get("cell_type") != "code":
        continue

    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)

    print(f"\n{'=' * 60}\nЯЧЕЙКА {number}: КОД\n{'=' * 60}")
    print(source)

    # Сохранённый текстовый вывод помогает проверить реальные
    # количества файлов и результаты очистки того запуска.
    streams = []
    for output in cell.get("outputs", []):
        if output.get("output_type") == "stream":
            text = output.get("text", "")
            if isinstance(text, list):
                text = "".join(text)
            streams.append(text)

    if streams:
        print(f"\n--- СОХРАНЁННЫЙ ВЫВОД ЯЧЕЙКИ {number} ---")
        print("".join(streams))

print("\nКонец чтения. Исходный код НЕ выполнялся.")
print("Обучение НЕ запускалось. Файлы не изменены.")

ИСХОДНЫЙ БЛОКНОТ: what_to_cook_training.ipynb
SHA-256: 6a556bcac98a14166785d4825dab5bd90afef4d6c62004c4ef3edc8894fc9775

ЯЧЕЙКА 1: КОД
from google.colab import drive
drive.mount('/content/drive')

--- СОХРАНЁННЫЙ ВЫВОД ЯЧЕЙКИ 1 ---
Mounted at /content/drive


ЯЧЕЙКА 2: КОД

from pathlib import Path

folder = Path("/content/drive/MyDrive/what_to_cook_ai")

if folder.exists():
    print("✅ Папка найдена!")

    for file in folder.iterdir():
        print("📁", file.name)
else:
    print("❌ Папка не найдена!")


--- СОХРАНЁННЫЙ ВЫВОД ЯЧЕЙКИ 2 ---
✅ Папка найдена!
📁 Food Ingredients Detection.v3i.yolov8.zip


ЯЧЕЙКА 3: КОД

from pathlib import Path
import zipfile

# Находим скачанный архив
folder = Path("/content/drive/MyDrive/what_to_cook_ai")
archive = folder / "Food Ingredients Detection.v3i.yolov8.zip"

# Папка, куда распакуем фотографии
dataset_folder = Path("/content/food_dataset")
dataset_folder.mkdir(exist_ok=True)

# Распаковываем архив
with zipfile.ZipFile(archive, "r") as zip_ref